# Working with Large Language Models

**Module 08: Large Language Models**  
**Objective**: Implement and use LLMs for practical applications

Practical LLM usage:
- Loading pretrained models
- Prompt engineering
- In-context learning
- Text generation strategies
- Model evaluation

## What You'll Learn
1. Using HuggingFace Transformers
2. Loading and running LLMs (GPT-2, LLaMA)
3. Prompt engineering techniques
4. Generation strategies (sampling, beam search, top-k, top-p)
5. In-context learning and few-shot prompting
6. Model quantization for efficiency

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)

## 1. Simple GPT Implementation

In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention."""
    
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = d_model // n_heads
        
        # Q, K, V projections
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        
        # Causal mask
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size))
                            .view(1, 1, block_size, block_size))
    
    def forward(self, x):
        B, T, C = x.size()  # batch, sequence length, d_model
        
        # Compute Q, K, V
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        
        # Reshape for multi-head attention
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, heads, T, d_k)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        # Attention scores
        att = (q @ k.transpose(-2, -1)) * (1.0 / np.sqrt(self.d_k))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        
        # Apply attention to values
        y = att @ v  # (B, heads, T, d_k)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        
        # Output projection
        y = self.resid_dropout(self.out_proj(y))
        
        return y

class GPTBlock(nn.Module):
    """Transformer block (GPT-style)."""
    
    def __init__(self, d_model, n_heads, d_ff, block_size, dropout=0.1):
        super().__init__()
        
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class SimpleGPT(nn.Module):
    """Simple GPT model."""
    
    def __init__(self, vocab_size, block_size=1024, n_layers=12, d_model=768, 
                 n_heads=12, d_ff=3072, dropout=0.1):
        super().__init__()
        
        self.block_size = block_size
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, block_size, dropout)
            for _ in range(n_layers)
        ])
        
        # Final layer norm and output
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying
        self.token_embedding.weight = self.lm_head.weight
        
        # Initialize weights
        self.apply(self._init_weights)
        
        print(f"GPT model: {sum(p.numel() for p in self.parameters())/1e6:.2f}M parameters")
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.block_size, f"Sequence length {T} exceeds block size {self.block_size}"
        
        # Embeddings
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)  # (1, T)
        tok_emb = self.token_embedding(idx)  # (B, T, d_model)
        pos_emb = self.position_embedding(pos)  # (1, T, d_model)
        x = self.dropout(tok_emb + pos_emb)
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        # Compute loss if targets provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        """
        Generate text autoregressively.
        
        Args:
            idx: (B, T) initial sequence
            max_new_tokens: number of tokens to generate
            temperature: sampling temperature
            top_k: top-k sampling
            top_p: nucleus (top-p) sampling
        """
        for _ in range(max_new_tokens):
            # Crop to block_size
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            
            # Forward
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            # Top-p (nucleus) sampling
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                
                # Remove tokens with cumulative probability above threshold
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                logits[indices_to_remove] = -float('Inf')
            
            # Sample
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            
            # Append
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx

# Create small GPT model
model = SimpleGPT(
    vocab_size=50257,  # GPT-2 vocab size
    block_size=256,
    n_layers=6,
    d_model=384,
    n_heads=6,
    d_ff=1536,
    dropout=0.1
).to(device)

# Test forward pass
test_input = torch.randint(0, 50257, (2, 10)).to(device)
logits, loss = model(test_input, test_input)

print(f"\nInput shape: {test_input.shape}")
print(f"Output logits shape: {logits.shape}")

## 2. Text Generation Strategies

In [ ]:
def compare_sampling_strategies():
    """Compare different sampling strategies."""
    
    # Create simple logits (for 10 tokens)
    torch.manual_seed(42)
    logits = torch.randn(1, 10)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    # 1. Greedy (argmax)
    probs_greedy = F.softmax(logits, dim=-1)
    selected_greedy = torch.argmax(logits, dim=-1).item()
    axes[0].bar(range(10), probs_greedy[0].numpy())
    axes[0].axvline(selected_greedy, color='red', linestyle='--', label='Selected')
    axes[0].set_title('Greedy Decoding (argmax)')
    axes[0].set_ylabel('Probability')
    axes[0].legend()
    
    # 2. Temperature = 0.5 (sharper)
    temp = 0.5
    probs_cold = F.softmax(logits / temp, dim=-1)
    axes[1].bar(range(10), probs_cold[0].numpy())
    axes[1].set_title(f'Temperature = {temp} (confident)')
    axes[1].set_ylabel('Probability')
    
    # 3. Temperature = 2.0 (flatter)
    temp = 2.0
    probs_hot = F.softmax(logits / temp, dim=-1)
    axes[2].bar(range(10), probs_hot[0].numpy())
    axes[2].set_title(f'Temperature = {temp} (diverse)')
    axes[2].set_ylabel('Probability')
    
    # 4. Top-k = 3
    top_k = 3
    v, _ = torch.topk(logits, top_k)
    logits_topk = logits.clone()
    logits_topk[logits < v[:, [-1]]] = -float('Inf')
    probs_topk = F.softmax(logits_topk, dim=-1)
    axes[3].bar(range(10), probs_topk[0].numpy())
    axes[3].set_title(f'Top-k = {top_k}')
    axes[3].set_ylabel('Probability')
    axes[3].set_xlabel('Token')
    
    # 5. Top-p = 0.8
    top_p = 0.8
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    logits_topp = logits.clone()
    indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
    logits_topp[indices_to_remove] = -float('Inf')
    probs_topp = F.softmax(logits_topp, dim=-1)
    axes[4].bar(range(10), probs_topp[0].numpy())
    axes[4].set_title(f'Top-p = {top_p} (nucleus)')
    axes[4].set_ylabel('Probability')
    axes[4].set_xlabel('Token')
    
    # 6. Combined: Temperature + Top-p
    logits_combined = logits / 1.2
    sorted_logits, sorted_indices = torch.sort(logits_combined, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > 0.9
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0
    indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
    logits_combined[indices_to_remove] = -float('Inf')
    probs_combined = F.softmax(logits_combined, dim=-1)
    axes[5].bar(range(10), probs_combined[0].numpy())
    axes[5].set_title('Temp=1.2 + Top-p=0.9')
    axes[5].set_ylabel('Probability')
    axes[5].set_xlabel('Token')
    
    plt.tight_layout()
    plt.show()
    
    print("\nSampling Strategies:")
    print("1. Greedy: Always pick most likely (deterministic, repetitive)")
    print("2. Temperature: Control randomness (low=confident, high=diverse)")
    print("3. Top-k: Sample from k most likely tokens")
    print("4. Top-p (nucleus): Sample from smallest set with cumulative prob > p")
    print("5. Combined: Temperature + Top-p (most common in practice)")

compare_sampling_strategies()

## 3. Prompt Engineering

In [ ]:
# Prompt engineering examples

prompt_examples = {
    "Zero-shot": {
        "description": "Direct task without examples",
        "prompt": "Translate to French: Hello, how are you?",
        "expected": "Bonjour, comment allez-vous?"
    },
    
    "Few-shot": {
        "description": "Provide examples before task",
        "prompt": """Translate to French:
        
English: Good morning
French: Bonjour

English: Thank you
French: Merci

English: Hello, how are you?
French:""",
        "expected": "Bonjour, comment allez-vous?"
    },
    
    "Chain-of-Thought": {
        "description": "Show reasoning steps",
        "prompt": """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: Roger started with 5 balls. 2 cans of 3 tennis balls each is 2×3=6 tennis balls. 5+6=11. The answer is 11.

Q: A juggler can juggle 16 balls. Half of the balls are golf balls, and half of the golf balls are blue. How many blue golf balls are there?
A:""",
        "expected": "Half of 16 is 8 golf balls. Half of 8 golf balls is 4 blue golf balls. The answer is 4."
    },
    
    "Role Prompting": {
        "description": "Assign specific role to model",
        "prompt": """You are a helpful Python programming tutor. Explain list comprehensions to a beginner.

Student: What is a list comprehension?
Tutor:""",
        "expected": "A list comprehension is a concise way to create lists in Python..."
    },
    
    "Instruction Following": {
        "description": "Clear instructions with format",
        "prompt": """Given a sentence, extract all named entities.

Sentence: "Apple Inc. was founded by Steve Jobs in Cupertino."

Output format:
- Organization: ...
- Person: ...
- Location: ...

Output:""",
        "expected": "- Organization: Apple Inc.\n- Person: Steve Jobs\n- Location: Cupertino"
    }
}

print("Prompt Engineering Techniques:\n")
print("="*80)

for technique, details in prompt_examples.items():
    print(f"\n{technique.upper()}")
    print(f"Description: {details['description']}")
    print(f"\nPrompt:\n{details['prompt'][:200]}...")
    print(f"\nExpected: {details['expected'][:100]}...")
    print("="*80)

## 4. In-Context Learning

In [ ]:
def demonstrate_in_context_learning():
    """Visualize in-context learning performance."""
    
    # Simulate accuracy vs number of examples
    n_examples = [0, 1, 2, 4, 8, 16, 32]
    
    # Different model sizes
    small_model = [25, 30, 35, 42, 48, 52, 54]  # Limited improvement
    medium_model = [30, 38, 48, 58, 68, 75, 78]  # Good improvement
    large_model = [35, 50, 65, 78, 85, 90, 92]  # Strong in-context learning
    
    plt.figure(figsize=(12, 6))
    
    plt.plot(n_examples, small_model, marker='o', linewidth=2, label='Small Model (1B)')
    plt.plot(n_examples, medium_model, marker='s', linewidth=2, label='Medium Model (13B)')
    plt.plot(n_examples, large_model, marker='^', linewidth=2, label='Large Model (70B)')
    
    plt.xlabel('Number of In-Context Examples', fontsize=12)
    plt.ylabel('Task Accuracy (%)', fontsize=12)
    plt.title('In-Context Learning: Performance vs Examples', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nIn-Context Learning Insights:")
    print("1. Larger models benefit more from examples")
    print("2. Returns diminish after ~8-16 examples")
    print("3. Small models plateau quickly")
    print("4. Emergent ability: Appears at scale (~10B+ parameters)")
    print("\n✓ Few-shot learning enables task adaptation without fine-tuning!")

demonstrate_in_context_learning()

## 5. Model Quantization

**Problem**: Large models consume too much memory

**Solution**: Reduce precision (FP32 → FP16 → INT8)

**Techniques**:
- Post-training quantization
- Quantization-aware training
- Mixed precision

In [ ]:
def compare_precision():
    """Compare memory usage across precisions."""
    
    # Model sizes (billions of parameters)
    model_sizes = [1, 7, 13, 30, 70, 175]
    
    # Memory per parameter (bytes)
    fp32_memory = [size * 4 for size in model_sizes]  # 4 bytes per param
    fp16_memory = [size * 2 for size in model_sizes]  # 2 bytes
    int8_memory = [size * 1 for size in model_sizes]  # 1 byte
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Memory comparison
    x = np.arange(len(model_sizes))
    width = 0.25
    
    axes[0].bar(x - width, fp32_memory, width, label='FP32', color='lightcoral')
    axes[0].bar(x, fp16_memory, width, label='FP16', color='lightblue')
    axes[0].bar(x + width, int8_memory, width, label='INT8', color='lightgreen')
    
    axes[0].set_xlabel('Model Size (B params)')
    axes[0].set_ylabel('Memory (GB)')
    axes[0].set_title('Memory Usage by Precision')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'{s}B' for s in model_sizes])
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Speedup vs accuracy tradeoff
    precision_types = ['FP32', 'FP16', 'INT8', 'INT4']
    speedup = [1.0, 1.8, 3.5, 6.0]
    accuracy_loss = [0, 0.1, 1.0, 3.5]
    
    axes[1].scatter(speedup, accuracy_loss, s=200, alpha=0.6, c=['red', 'blue', 'green', 'orange'])
    
    for i, label in enumerate(precision_types):
        axes[1].annotate(label, (speedup[i], accuracy_loss[i]), 
                        xytext=(10, 5), textcoords='offset points', fontsize=11)
    
    axes[1].set_xlabel('Inference Speedup (×)', fontsize=12)
    axes[1].set_ylabel('Accuracy Loss (%)', fontsize=12)
    axes[1].set_title('Precision Trade-off: Speed vs Accuracy')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(0, 7)
    axes[1].set_ylim(-0.5, 4)
    
    plt.tight_layout()
    plt.show()
    
    print("\nQuantization Insights:")
    print("1. FP16: 2× memory reduction, minimal accuracy loss")
    print("2. INT8: 4× memory reduction, ~1% accuracy loss")
    print("3. INT4: 8× memory reduction, ~3-5% accuracy loss")
    print("4. Enables running 70B models on consumer GPUs")
    print("\n✓ Quantization makes large models accessible!")

compare_precision()

## 6. HuggingFace API Overview

In [ ]:
# HuggingFace Transformers API examples

huggingface_examples = """
# ========================================
# Loading Pretrained Models
# ========================================

from transformers import AutoTokenizer, AutoModelForCausalLM

# GPT-2
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

# GPT-Neo (larger)
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-2.7B")

# LLaMA (requires access)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")


# ========================================
# Text Generation
# ========================================

# Basic generation
inputs = tokenizer("The future of AI is", return_tensors="pt")
outputs = model.generate(**inputs, max_length=50)
text = tokenizer.decode(outputs[0])

# With sampling parameters
outputs = model.generate(
    **inputs,
    max_length=100,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    num_return_sequences=3  # Generate 3 variants
)


# ========================================
# Pipeline API (High-level)
# ========================================

from transformers import pipeline

# Text generation
generator = pipeline('text-generation', model='gpt2')
outputs = generator("Once upon a time", max_length=50)

# Other pipelines
classifier = pipeline('sentiment-analysis')
qa_model = pipeline('question-answering')
summarizer = pipeline('summarization')


# ========================================
# 8-bit Quantization (Reduce memory)
# ========================================

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    load_in_8bit=True,
    device_map="auto"  # Automatically distribute across GPUs
)


# ========================================
# Custom Generation Configuration
# ========================================

from transformers import GenerationConfig

gen_config = GenerationConfig(
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.2,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

outputs = model.generate(**inputs, generation_config=gen_config)


# ========================================
# Batched Generation (Efficient)
# ========================================

prompts = ["Hello", "The weather is", "AI will"]
inputs = tokenizer(prompts, return_tensors="pt", padding=True)
outputs = model.generate(**inputs, max_length=30)

for i, output in enumerate(outputs):
    print(f"Prompt {i}: {tokenizer.decode(output)}")


# ========================================
# Streaming Generation (Real-time)
# ========================================

from transformers import TextIteratorStreamer
from threading import Thread

streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)

generation_kwargs = dict(**inputs, streamer=streamer, max_length=100)
thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

for new_text in streamer:
    print(new_text, end="", flush=True)

thread.join()
"""

print("HuggingFace Transformers API Examples:\n")
print(huggingface_examples)

## 7. Model Evaluation

In [ ]:
def compare_evaluation_metrics():
    """Compare different LLM evaluation approaches."""
    
    metrics = {
        'Perplexity': {
            'description': 'How surprised is model by test data',
            'formula': 'exp(cross_entropy_loss)',
            'lower_is_better': True,
            'use_case': 'Language modeling quality'
        },
        'BLEU': {
            'description': 'N-gram overlap with references',
            'formula': 'Precision of n-grams',
            'lower_is_better': False,
            'use_case': 'Translation, generation'
        },
        'ROUGE': {
            'description': 'Recall-oriented n-gram matching',
            'formula': 'Recall of n-grams',
            'lower_is_better': False,
            'use_case': 'Summarization'
        },
        'BERTScore': {
            'description': 'Semantic similarity using embeddings',
            'formula': 'Cosine similarity of embeddings',
            'lower_is_better': False,
            'use_case': 'Semantic evaluation'
        },
        'Human Eval': {
            'description': 'Human judges rate outputs',
            'formula': 'Average human rating',
            'lower_is_better': False,
            'use_case': 'Gold standard (expensive)'
        }
    }
    
    print("\nLLM Evaluation Metrics:\n")
    print("="*80)
    
    for metric, details in metrics.items():
        print(f"\n{metric.upper()}")
        print(f"  Description: {details['description']}")
        print(f"  Formula: {details['formula']}")
        print(f"  Direction: {'Lower is better' if details['lower_is_better'] else 'Higher is better'}")
        print(f"  Use case: {details['use_case']}")
    
    print("\n" + "="*80)
    
    # Visualize typical scores
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Compare models on different metrics
    models = ['GPT-2', 'GPT-3', 'GPT-4', 'LLaMA-70B']
    perplexity = [35, 20, 15, 18]  # Lower is better
    human_rating = [6.5, 7.8, 8.9, 8.2]  # Out of 10
    
    x = np.arange(len(models))
    axes[0].bar(x, perplexity, color='lightcoral')
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('Perplexity (lower=better)')
    axes[0].set_title('Perplexity Comparison')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(models)
    axes[0].grid(axis='y', alpha=0.3)
    
    axes[1].bar(x, human_rating, color='lightblue')
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('Human Rating (out of 10)')
    axes[1].set_title('Human Evaluation')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(models)
    axes[1].set_ylim(0, 10)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

compare_evaluation_metrics()

## Summary

You've mastered working with LLMs:
- ✅ Implemented simple GPT from scratch
- ✅ Text generation strategies (temperature, top-k, top-p)
- ✅ Prompt engineering techniques
- ✅ In-context learning
- ✅ Model quantization for efficiency
- ✅ HuggingFace Transformers API
- ✅ Evaluation metrics

**Key insights**:
1. **Generation**: Temperature + top-p most common
2. **Prompting**: Few-shot > zero-shot for complex tasks
3. **Quantization**: 8-bit enables 2-4× memory reduction
4. **Evaluation**: Perplexity + human eval most reliable
5. **Scale matters**: Larger models show qualitatively different behavior

**Best practices**:
1. Start with pretrained models (don't train from scratch)
2. Use prompt engineering before fine-tuning
3. Apply quantization for deployment
4. Batch requests for efficiency
5. Cache when possible (KV cache, response cache)

**Next Module**: Fine-tuning LLMs (LoRA, QLoRA, PEFT)

## Exercises

1. **Implement Beam Search**: Add beam search generation to SimpleGPT
2. **Prompt Optimization**: Test different prompt formats on same task
3. **Quantization**: Implement simple post-training quantization (FP32 → INT8)